In [ ]:
import sqlite3
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [14]:
# Loading dataset and prepping features
conn = sqlite3.connect('dbs/song_features.db')
song_features_df = pd.read_sql_query("SELECT * FROM song_features", conn)
conn.close()

X = song_features_df[['NORM_in_number_of_playlists', 'NORM_avg_playlist_followers', 'NORM_avg_position_in_playlist']]
Y = song_features_df['cluster']

In [15]:
# Splitting training/testing data with stratification (equal distribution of clusters).
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

In [ ]:
# Training decision tree classifier
clf = DecisionTreeClassifier(class_weight='balanced', random_state=42) # attempting to balance the classes
clf.fit(X_train, Y_train)

DecisionTreeClassifier(class_weight='balanced', random_state=42)

In [ ]:
# Model evaluation
Y_pred = clf.predict(X_test)
print("Classification Report:")
print(classification_report(Y_test, Y_pred))
print("Confusion Matrix:")
print(confusion_matrix(Y_test, Y_pred))
# There is an issue here: due to the imbalance of cluster 0 and 1 vs 2 and 3 with a split of 1713638 / 545869 / 2035 / 750 in the dataset,
# the model is predicting cluster 0 and 1 almost exclusively, with very few predictions for clusters 2 and 3. This leads to an artificially high accuracy
# since even guessing between only cluster 0 or 1 would yield a high accuracy score. As such, the model will likely perform poorly on the minority classes on a larger scale.
# We will attempt to address this by using a one vs. all model strategy to train a separate model for each cluster.

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    342728
           1       1.00      1.00      1.00    109174
           2       1.00      1.00      1.00       407
           3       1.00      1.00      1.00       150

    accuracy                           1.00    452459
   macro avg       1.00      1.00      1.00    452459
weighted avg       1.00      1.00      1.00    452459

Confusion Matrix:
[[342721      7      0      0]
 [     4 109170      0      0]
 [     0      0    407      0]
 [     0      0      0    150]]


In [20]:
# After some research, we will use a one vs. all model strategy to train a separate model for each cluster.
# Due to the low number of clusters and their relatively clear-cut differences, this should be efficient and effective.
import sqlite3
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Preparing features for clustering
features = ['NORM_in_number_of_playlists', 'NORM_avg_playlist_followers', 'NORM_avg_position_in_playlist']
X = song_features_df[features]
Y = song_features_df['cluster']

In [21]:
# Splitting training/testing data with stratification (equal distribution of clusters).
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

clusters = Y.unique()
classifiers = {}
predictions = pd.DataFrame(index=X_test.index)

for cluster in clusters:
    Y_train_bin = (Y_train == cluster).astype(int)
    clf = DecisionTreeClassifier(class_weight='balanced', random_state=42)
    clf.fit(X_train, Y_train_bin)
    classifiers[cluster] = clf

    prediction_probability = clf.predict_proba(X_test)[:, 1]
    predictions[cluster] = prediction_probability

final_predictions = predictions.idxmax(axis=1)

In [ ]:
# Model evaluation
print("Classification Report:")
print(classification_report(Y_test, final_predictions))
print("Confusion Matrix:")
print(confusion_matrix(Y_test, final_predictions))
# The models still did a superb job of predicting every cluster, with almost exactly the same predictions as the original model.
# However, given the structure of our dataset, this is expected. The new one vs. all model strategy is more robust by design, though, and will likely perform better on a larger scale.
# In a positive light, the one vs. all model has highlighted how strong the differences are between the clusters, as the models were able to predict them so accurately.

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    342728
           1       1.00      1.00      1.00    109174
           2       1.00      1.00      1.00       407
           3       1.00      1.00      1.00       150

    accuracy                           1.00    452459
   macro avg       1.00      1.00      1.00    452459
weighted avg       1.00      1.00      1.00    452459

Confusion Matrix:
[[342721      6      1      0]
 [     4 109170      0      0]
 [     0      0    407      0]
 [     0      0      0    150]]
